# NOTEBOOK 05 – VĨNH CHI | CHƯƠNG 5: KẾT QUẢ & THẢO LUẬN

**Phụ trách:** 5.1–5.4 · GitHub Issue **#6** · Bảng kết quả Test hạn 24/11, Ch5 hoàn chỉnh 25/11/2026.

**Vai trò độc lập:** đánh giá lại cả Random Forest (Ngọc Lan), Dummy (Trúc Chi) và MLP (Tấn Lên) trên **cùng frozen Test**, đối soát các artifact mà không fit lại.

**Nhận từ Ngọc Lan:** RF pipeline/metadata Validation. **Nhận từ Trúc Chi:** `test_predictions.csv`, `experiment_metadata.json` và thông tin ngưỡng/cấu hình chọn trên Validation.

**Bàn giao Tấn Lên:** `reports/tables/model_metrics.csv`, hình PR Curve, Top-K và bản phân tích lỗi.

[Lý thuyết Ch5](../docs/theory/05_vinh_chi_ch5.md) · [Giao thức thực nghiệm](../docs/EXPERIMENT_PROTOCOL.md).

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_project_root():
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "docs" / "DATA_CONTRACT.md").exists():
            return candidate
    raise FileNotFoundError("Mở notebook trong repository (hoặc thư mục notebooks/).")
ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)

## 5.1 Kiểm tra frozen Test và lineage của cả 3 mô hình

Vĩnh Chi **không huấn luyện hoặc tối ưu ngưỡng**. Thay vào đó:
- Đối chiếu `split_id`, số đơn và thứ tự `order_id` giữa manifest với Test predictions Ch4.
- Load RF do Ngọc Lan fit, Dummy do Trúc Chi fit và MLP do Tấn Lên fit (chỉ artifact nội bộ đã xác minh).
- **Tự gọi predict_proba(Test)** cho cả ba và so sánh với `test_predictions.csv` để phát hiện kết quả bàn giao không khớp.
- Tính lại metrics độc lập; khác biệt phải báo Tấn Lên trước khi làm báo cáo.

In [ ]:
import json, joblib
from sklearn.metrics import (
    average_precision_score, confusion_matrix,
    precision_score, recall_score, f1_score
)
from src.experiment import get_shared_splits, SAFE_FEATURES
pred_file = PROCESSED / "test_predictions.csv"
meta_file = PROCESSED / "experiment_metadata.json"
if not pred_file.exists() or not meta_file.exists():
    raise FileNotFoundError("Thiếu frozen Test và metadata: yêu cầu Trúc Chi bàn giao.")
pred = pd.read_csv(pred_file)
meta = json.loads(meta_file.read_text(encoding="utf-8"))
print("=== KIỂM TRA MÔ HÌNH CHÍNH ===")
print("Champion:", meta.get("champion"))
print("Ngưỡng Champion:", meta.get("chosen_threshold"))
print("Ngưỡng các mô hình:", meta.get("thresholds_from_val"))
print("AP trên Validation:", meta.get("validation_AP"))
splits = get_shared_splits(PROCESSED)
test_frame = splits["test"]
assert meta["split_id"] == splits["metadata"]["split_id"]
assert meta["features"] == list(SAFE_FEATURES)
assert meta["model_owners"] == {
    "rf": "Ngọc Lan", "dummy": "Trúc Chi", "mlp": "Tấn Lên"
}
expected = {"order_id","y_true","p_dummy","p_rf","p_mlp","y_pred_champion"}
assert expected.issubset(pred.columns), expected.difference(pred.columns)
assert pred["order_id"].is_unique and len(pred) == len(test_frame)
assert np.array_equal(pred["order_id"].to_numpy(),
                      test_frame["order_id"].to_numpy()), "Sai thứ tự/mẫu Test"
assert np.array_equal(pred["y_true"].to_numpy().astype(int),
                      test_frame["is_late"].to_numpy().astype(int))
for name in ("dummy", "rf", "mlp"):
    assert pred["p_"+name].between(0,1).all()
print("Split, sample order_id và model owner hợp lệ:", meta["split_id"])
print("Số dòng Test:", len(pred))

champion = meta["champion"]
threshold = float(meta["chosen_threshold"])

assert champion == "rf", (
    "Champion không khớp quyết định RF của nhóm!"
)

assert np.isclose(
    threshold,
    float(meta["thresholds_from_val"][champion])
)

y_recomputed = (
    pred["p_rf"].to_numpy() >= threshold
).astype(int)

y_saved = pred["y_pred_champion"].to_numpy().astype(int)

assert np.array_equal(y_recomputed, y_saved), (
    "Nhãn Champion không khớp xác suất RF và threshold!"
)

print("PASS: Nhãn Champion RF nhất quán với Validation.")

## 5.1.1 Kiểm định dự đoán bằng artifact độc lập (KHÔNG FIT)

Đây là bước kiểm định bổ sung: Vĩnh Chi dùng model do Ngọc Lan và Trúc Chi bàn giao để tính lại xác suất từ Test. Nếu kết quả không trùng \`test_predictions.csv\`, không được công bố kết quả cho tới khi giải quyết.

In [ ]:
model_files = {
    "rf": ROOT / "models" / "ngoc_lan_random_forest.joblib",
    "dummy": ROOT / "models" / "dummy_truc_chi.joblib",
    "mlp": ROOT / "models" / "tan_len_mlp.joblib",
}
missing = [p.name for p in model_files.values() if not p.exists()]
if missing:
    raise FileNotFoundError("Thiếu artifact model cần kiểm định: " + ", ".join(missing))
X_test = test_frame[list(SAFE_FEATURES)]
for name, path in model_files.items():
    model = joblib.load(path)  # Chỉ load artifact tin cậy từ nội bộ nhóm
    rescored = model.predict_proba(X_test)[:,1]
    assert np.allclose(
        pred["p_"+name].to_numpy(), rescored, atol=1e-7
    ), f"Model {name} không khớp predictions bàn giao"
print("Vĩnh Chi đã tính lại xác suất của RF, Dummy và MLP trên frozen Test.")

## 5.1.1 Bảng chỉ số và ma trận nhầm lẫn

**Diễn giải:** Precision gắn với cảnh báo sai, Recall gắn với đơn trễ bị bỏ sót, F1 cân bằng hai khía cạnh, AP đo độ tốt xếp hạng qua ngưỡng. Luôn báo **số lượng thực (TP/FP/FN/TN)**, không chỉ phần trăm.

Ngưỡng của mỗi mô hình phải là ngưỡng đã chọn từ Validation; Dummy là mức so sánh.

In [ ]:
actual = pred["y_true"].to_numpy().astype(int)
rows = []
for name in ["dummy", "rf", "mlp"]:
    probabilities = pred["p_"+name].to_numpy()
    threshold = float(meta["thresholds_from_val"][name])
    labels = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(actual, labels, labels=[0,1]).ravel()
    rows.append({
        "model":name, "threshold_val":threshold, "AP":average_precision_score(actual, probabilities),
        "precision":precision_score(actual, labels, zero_division=0),
        "recall":recall_score(actual, labels, zero_division=0),
        "f1":f1_score(actual, labels, zero_division=0),
        "TP":int(tp), "FP":int(fp), "FN":int(fn), "TN":int(tn), "test_n":len(actual)
    })
metrics = pd.DataFrame(rows).sort_values("AP", ascending=False)
display(metrics)
OUT_TABLES = ROOT / "reports" / "tables"
OUT_TABLES.mkdir(parents=True, exist_ok=True)
metrics.to_csv(OUT_TABLES / "model_metrics.csv", index=False)
print("Bảng metric đã xuất (chỉ có giá trị thực nếu Ch4 đã chạy).")

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay
fig, ax = plt.subplots(figsize=(7,5))
for name in ["dummy","rf","mlp"]:
    PrecisionRecallDisplay.from_predictions(actual, pred["p_"+name], name=name, ax=ax)
ax.set_title("Đường Precision–Recall trên Test – Olist")
ax.grid(alpha=0.2)
fig.tight_layout()
OUT_FIGS = ROOT / "reports" / "figures"
OUT_FIGS.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT_FIGS / "pr_curve_test.png", dpi=160)
plt.show()

## 5.2 Phân tích lỗi theo phân khúc

Cần chỉ ra khu vực, giá trị đơn, người bán/nhóm hàng có những cảnh báo sai hoặc bị bỏ sót. Chỉ phân tích slice có **đủ số mẫu**, tránh rút kết luận mạnh từ 2–3 đơn.

Có thể nối Test predictions với file `orders_model_ready.csv` bằng `order_id` (validate one_to_one), nhưng không coi các cột hậu nghiệm thành input của model.

In [ ]:

from src.contracts import assert_unique_orders

source = PROCESSED / "orders_model_ready.csv"

if not source.exists():
    raise FileNotFoundError(
        "Thiếu orders_model_ready.csv của Chương 3."
    )

features = pd.read_csv(source)
assert_unique_orders(features)

champion = meta["champion"]

assert champion == "rf", (
    "Champion không phải Random Forest!"
)

threshold = float(meta["thresholds_from_val"]["rf"])

assert np.isclose(
    threshold,
    float(meta["chosen_threshold"])
)

pred_eval = pred.copy()

pred_eval["chosen_pred"] = (
    pred_eval["p_rf"] >= threshold
).astype(int)

required_cols = [
    "order_id",
    "customer_state",
    "total_price",
    "seller_count"
]

missing_cols = [
    c for c in required_cols
    if c not in features.columns
]

assert not missing_cols, (
    f"Model-ready thiếu cột: {missing_cols}"
)

joined = pred_eval.merge(
    features[required_cols],
    on="order_id",
    how="left",
    validate="one_to_one",
    indicator=True
)

assert len(joined) == len(pred_eval)
assert joined["_merge"].eq("both").all()

joined = joined.drop(columns=["_merge"])

for col in [
    "customer_state",
    "total_price",
    "seller_count"
]:
    assert joined[col].notna().all(), (
        f"Có đơn Test thiếu {col}"
    )

print("PASS: Phép nối dữ liệu hợp lệ.")
print("Số đơn Test:", len(pred_eval))
print("Số đơn sau khi nối:", len(joined))

joined["TP"] = (
    (joined["y_true"] == 1) &
    (joined["chosen_pred"] == 1)
).astype(int)

joined["FP"] = (
    (joined["y_true"] == 0) &
    (joined["chosen_pred"] == 1)
).astype(int)

joined["FN"] = (
    (joined["y_true"] == 1) &
    (joined["chosen_pred"] == 0)
).astype(int)

joined["TN"] = (
    (joined["y_true"] == 0) &
    (joined["chosen_pred"] == 0)
).astype(int)

rf_metrics = metrics.set_index("model").loc["rf"]

for col in ["TP", "FP", "FN", "TN"]:
    observed = int(joined[col].sum())
    expected = int(rf_metrics[col])

    assert observed == expected, (
        f"{col} không khớp: "
        f"{observed} != {expected}"
    )

print("PASS: TP, FP, FN, TN khớp RF Test.")

joined["order_value_group"] = pd.qcut(
    joined["total_price"],
    q=4,
    duplicates="drop"
).astype(str)

joined["seller_group"] = np.where(
    joined["seller_count"] > 1,
    "Nhiều seller (>1)",
    "1 seller"
)


weight_col = None

for col in [
    "total_weight_g",
    "product_weight_g_sum",
    "avg_weight_g"
]:
    if col in features.columns:
        weight_col = col
        break

if weight_col is not None:

    weight_data = features[
        ["order_id", weight_col]
    ].copy()

    joined = joined.merge(
        weight_data,
        on="order_id",
        how="left",
        validate="one_to_one"
    )

    print("Cột trọng lượng:", weight_col)

else:
    items_file = (
        RAW / "olist_order_items_dataset.csv"
    )

    products_file = (
        RAW / "olist_products_dataset.csv"
    )

    if not items_file.exists() or not products_file.exists():
        raise FileNotFoundError(
            "Thiếu dữ liệu gốc để tính trọng lượng."
        )

    items = pd.read_csv(
        items_file,
        usecols=["order_id", "product_id"]
    )

    products = pd.read_csv(
        products_file,
        usecols=["product_id", "product_weight_g"]
    )

    assert products["product_id"].is_unique

    items_weight = items.merge(
        products,
        on="product_id",
        how="left",
        validate="many_to_one"
    )

    items_weight["product_weight_g"] = pd.to_numeric(
        items_weight["product_weight_g"],
        errors="coerce"
    )

    order_weight = (
        items_weight.groupby("order_id")[
            "product_weight_g"
        ]
        .agg(lambda s: s.sum(min_count=len(s)))
        .rename("total_weight_g")
        .reset_index()
    )

    joined = joined.merge(
        order_weight,
        on="order_id",
        how="left",
        validate="one_to_one"
    )

    weight_col = "total_weight_g"
    print("Đã tổng hợp trọng lượng từ dữ liệu raw.")

weight_values = pd.to_numeric(
    joined[weight_col],
    errors="coerce"
)

valid_weight = (
    weight_values.notna() &
    (weight_values >= 0)
)

print("Đơn có trọng lượng hợp lệ:",
      int(valid_weight.sum()))
print("Đơn thiếu trọng lượng:",
      int((~valid_weight).sum()))

assert valid_weight.sum() >= 30
assert weight_values[valid_weight].nunique() >= 2

joined["weight_group"] = "Thiếu/không hợp lệ"

weight_bins = pd.qcut(
    weight_values.loc[valid_weight],
    q=4,
    duplicates="drop"
)

joined.loc[valid_weight, "weight_group"] = (
    weight_bins.astype(str)
)

print("Số nhóm trọng lượng hợp lệ:",
      len(weight_bins.cat.categories))

def summarize_error(df, group_col, min_n=30):

    summary = (
        df.groupby(
            group_col,
            dropna=False
        )
        .agg(
            n=("order_id", "size"),
            actual_late=("y_true", "sum"),
            TP=("TP", "sum"),
            FP=("FP", "sum"),
            FN=("FN", "sum"),
            TN=("TN", "sum")
        )
        .reset_index()
    )

   
    if group_col == "weight_group":
        summary = summary[
        (summary["n"] >= min_n) |
        (summary[group_col] == "Thiếu/không hợp lệ")
    ].copy()
    else:
        summary = summary[
        summary["n"] >= min_n
    ].copy()


    summary["late_rate"] = (
        summary["actual_late"] / summary["n"]
    )

    neg = summary["FP"] + summary["TN"]

    summary["FPR"] = (
        summary["FP"] / neg.replace(0, np.nan)
    )

    pos = summary["FN"] + summary["TP"]

    summary["FNR"] = (
        summary["FN"] / pos.replace(0, np.nan)
    )

    predicted_pos = (
        summary["TP"] + summary["FP"]
    )

    summary["precision"] = (
        summary["TP"] /
        predicted_pos.replace(0, np.nan)
    )

    summary["recall"] = (
        summary["TP"] / pos.replace(0, np.nan)
    )

    summary.insert(0, "dimension", group_col)

    return summary.sort_values(
        "n",
        ascending=False
    )

group_columns = [
    "customer_state",
    "order_value_group",
    "weight_group",
    "seller_group"
]

segment_tables = []

for group_col in group_columns:

    result = summarize_error(
        joined,
        group_col,
        min_n=30
    )

    print(
        f"\n=== Phân tích lỗi: {group_col} ==="
    )

    display(result)

    segment_tables.append(result.rename(
        columns={group_col: "group_value"}
    ))

error_segments = pd.concat(
    segment_tables,
    ignore_index=True
)

OUT_TABLES.mkdir(
    parents=True,
    exist_ok=True
)

error_segments.to_csv(
    OUT_TABLES / "error_analysis_segments.csv",
    index=False,
    encoding="utf-8-sig"
)

assert set(error_segments["dimension"]) == set(
    group_columns
)

assert len(joined) == len(pred_eval)

print("\nPASS: Đã phân tích đủ 4 nhóm.")
weight_summary = error_segments[
    error_segments["dimension"] == "weight_group"
]

assert weight_summary["n"].sum() == len(pred), (
    "Phân tích trọng lượng chưa bao phủ toàn bộ Test!"
)

print("PASS: Đã thống kê đủ đơn hàng theo trọng lượng.")
print("Mô hình:", champion)
print("Ngưỡng Validation:", threshold)
print("Số đơn Test:", len(joined))
print(
    "Đã lưu: reports/tables/"
    "error_analysis_segments.csv"
)


## 5.3 Đánh giá ý nghĩa vận hành: Top-K

Kịch bản: đội vận hành đủ khả năng theo dõi K đơn, cần chọn đơn có xác suất cao nhất. Tính **Precision@K**, **Recall@K**, **FP@K** và so sánh với cảnh báo theo ngưỡng.

K phải được chọn từ nhu cầu vận hành hoặc Validation; minh họa các giá trị K trên Test thì **chỉ để mô tả**, không dùng để tối ưu mô hình.

In [ ]:

from src.experiment import precision_recall_at_k

actual = pred["y_true"].to_numpy().astype(int)

test_n = len(actual)
total_late = int(actual.sum())
late_rate = total_late / test_n

k_values = [
    k for k in [50, 100, 200, 500]
    if 0 < k <= test_n
]

topk_rows = []

for model_name in ["rf", "mlp", "dummy"]:

    probs = pred["p_" + model_name].to_numpy()

    for k in k_values:

        result = precision_recall_at_k(
            actual,
            probs,
            k
        )

        result.update({
            "model": model_name,
            "pct_orders_checked": 100 * k / test_n,
            "Lift_at_K": (
                result["Precision_at_K"] / late_rate
                if late_rate > 0 else np.nan
            )
        })

        topk_rows.append(result)

topk = pd.DataFrame(topk_rows)

model_order = {
    "rf": 0,
    "mlp": 1,
    "dummy": 2
}

topk["_model_order"] = (
    topk["model"].map(model_order)
)

topk = (
    topk.sort_values(
        ["_model_order", "K"]
    )
    .drop(columns=["_model_order"])
    .reset_index(drop=True)
)

display(topk)

OUT_TABLES.mkdir(
    parents=True,
    exist_ok=True
)

topk.to_csv(
    OUT_TABLES / "topk_analysis.csv",
    index=False
)

print("PASS: Đã đánh giá Top-K cho RF, MLP, Dummy.")
print("Số đơn Test:", test_n)
print("Tổng đơn giao trễ:", total_late)
print("Đã lưu: reports/tables/topk_analysis.csv")


## 5.4 Thảo luận – so sánh mô hình & hạn chế

Cần trả lời:
- RF (Ngọc Lan) vs MLP (Tấn Lên) khác nhau thế nào về AP, Precision, Recall, F1 và FP? Chênh lệch có đáng kể trong thực tiễn không?
- Số cảnh báo sai của từng mô hình thay đổi như thế nào khi ưu tiên Top-K?
- Model mắc lỗi nhiều ở nhóm nào? Có thể do ít dữ liệu hoặc đặc trưng không đầy đủ?
- Những hạn chế: **delivered-only selection bias**, dữ liệu cũ (2016–2018), địa lý Brazil, không tự áp dụng kết quả cho sàn Việt Nam 2026, feature availability chưa xác minh hết.

**Không diễn giải AP là tỷ lệ phần trăm đơn đúng.** AP đo chất lượng xếp hạng Precision–Recall.

## 5.5 Checklist bàn giao
- [ ] Metrics Test cho Dummy/RF/MLP, có hỗ trợ số mẫu và nhãn.
- [ ] Confusion matrix và PR Curve từ dữ liệu thật.
- [ ] FP, FN, Precision@K, Recall@K và phân tích trade-off.
- [ ] Có bảng lỗi theo nhóm đủ kích thước mẫu, nhận xét trung lập.
- [ ] Có liên kết repo/commit của file dự đoán và ngưỡng Validation.
- [ ] Hoàn thành `reports/chapters/CH05_RESULTS.md`, gửi bảng/hình cho Tấn Lên.

**Cách trình bày:** cho thấy bảng kết quả trước, giải thích chỉ số theo góc nhìn vận hành, rồi phân tích sai và hạn chế; tránh chỉ khen mô hình có AP cao nhất.
- [ ] Đã tái dự đoán Test từ RF Ngọc Lan + Dummy Trúc Chi và MLP Tấn Lên và đối chiếu xác suất với file bàn giao.
- [ ] Top-K của cả 3 mô hình đã tính độc lập, không lựa chọn threshold bằng Test.